In [0]:
%sql

select * from data_modelling.silver.silver_table

order_id,order_date,customer_id,customer_name,customer_email,product_id,product_name,product_category,quantity,unit_price,payment_type,country,last_updated,customer_name_upper,process_date
1001,2024-07-01,1,Alice Johnson,alice@gmail.com,501,iPhone 14,Electronics,1,999.99,Credit Card,USA,2024-07-01,ALICE JOHNSON,2026-02-05
1002,2024-07-01,2,Bob Smith,bob@yahoo.com,502,AirPods Pro,Electronics,2,199.99,PayPal,USA,2024-07-01,BOB SMITH,2026-02-05
1003,2024-07-01,3,Charlie Brown,charlie@outlook.com,503,Nike Shoes,Footwear,1,129.99,Credit Card,Canada,2024-07-01,CHARLIE BROWN,2026-02-05


##### **Dim_Customer**

In [0]:
%sql

create or replace table data_modelling.gold.Dim_Customer
as
with rem_dup as (
    select DISTINCT customer_id,
           customer_name,
           customer_email,
           customer_name_upper
    from data_modelling.silver.silver_table
)
select customer_id,
       customer_name,
       customer_email,
       customer_name_upper,
       row_number() over(order by customer_id) as dim_cust_key
from rem_dup

num_affected_rows,num_inserted_rows


##### **Dim_Product**

In [0]:
%sql

create or replace table data_modelling.gold.Dim_Product
as
select DISTINCT product_id as dim_product_key,
          product_name,
          product_category
  from data_modelling.silver.silver_table

num_affected_rows,num_inserted_rows


##### **Dim_Payment**

In [0]:
%sql
create or replace table data_modelling.gold.Dim_Payment
as
with payment as (
select DISTINCT payment_type
  from data_modelling.silver.silver_table
)
select payment_type,
       row_number() over(order by payment_type) as dim_payment_key
from payment

num_affected_rows,num_inserted_rows


##### **Dim_Region**

In [0]:
%sql
create or replace table data_modelling.gold.Dim_Region
as
with region_name as (
select DISTINCT country
  from data_modelling.silver.silver_table
)
select country,
       row_number() over(order by country) as dim_country_key
from region_name

num_affected_rows,num_inserted_rows


##### **Dim_Sales**

In [0]:
%sql
create or replace table data_modelling.gold.Dim_Sales
as
select
    row_number() over(order by order_id) as dim_sales_key,
    order_id,
    order_date,
    customer_id,
    customer_name,
    customer_email,
    product_id,
    product_name,
    product_category,
    quantity,
    unit_price,
    payment_type,
    country,
    last_updated,
    customer_name_upper,
    process_date
from data_modelling.silver.silver_table

num_affected_rows,num_inserted_rows


##### **Fact_Orders**

In [0]:
%sql
create or replace table data_modelling.gold.Fact_Orders
as
select dim_sales_key,
        dim_cust_key,
        dim_product_key,
        dim_country_key,
        dim_payment_key,
        order_date,
        quantity,
        unit_price,
        (quantity * unit_price) as total_price 
from data_modelling.gold.Dim_Sales s
left join data_modelling.gold.Dim_customer on s.customer_id = dim_customer.customer_id
left join data_modelling.gold.Dim_Product on s.product_id = Dim_Product.dim_product_key
left join data_modelling.gold.Dim_Payment on s.payment_type = Dim_Payment.payment_type
left join data_modelling.gold.Dim_Region on s.country = Dim_Region.country




num_affected_rows,num_inserted_rows
